# Phoenix Urban Heat Island Mapper
Run each cell top to bottom (tap the ▶ button on each cell). Works entirely in the browser — no local setup needed.

**Steps:** install deps → fetch real Landsat + census data → run LST pipeline → zonal stats → vulnerability index + hotspot clustering → download results.

## 1. Install dependencies

In [ ]:
!pip install -q pystac-client planetary-computer rasterio geopandas rasterstats libpysal esda requests

## 2. Fetch a real Landsat 8/9 scene over Phoenix (Microsoft Planetary Computer, free, no auth)

In [ ]:
from pystac_client import Client
import planetary_computer as pc

PHOENIX_BBOX = [-112.20, 33.30, -111.90, 33.60]

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=PHOENIX_BBOX,
    datetime="2024-06-01/2024-08-31",
    query={"eo:cloud_cover": {"lt": 10}, "platform": {"in": ["landsat-8", "landsat-9"]}},
)
items = list(search.items())
print(f"Found {len(items)} low-cloud scenes")

item = min(items, key=lambda i: i.properties.get("eo:cloud_cover", 100))
signed = pc.sign(item)
print(f"Selected: {item.id}, cloud cover: {item.properties.get('eo:cloud_cover')}%, date: {item.properties.get('datetime')}")

band10_url = signed.assets["lwir11"].href
red_url = signed.assets["red"].href
nir_url = signed.assets["nir08"].href

## 3. Download bands, clip to Phoenix AOI

In [ ]:
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
import numpy as np

def read_clipped(url, bbox_wgs84):
    with rasterio.open(url) as src:
        bounds_native = transform_bounds("EPSG:4326", src.crs, *bbox_wgs84)
        window = from_bounds(*bounds_native, transform=src.transform)
        data = src.read(1, window=window)
        transform = src.window_transform(window)
        profile = src.profile.copy()
        profile.update(height=data.shape[0], width=data.shape[1], transform=transform)
        return data, profile

band10_dn, profile10 = read_clipped(band10_url, PHOENIX_BBOX)
red, profile_red = read_clipped(red_url, PHOENIX_BBOX)
nir, profile_nir = read_clipped(nir_url, PHOENIX_BBOX)

print("Band 10 shape:", band10_dn.shape, "| Red shape:", red.shape, "| NIR shape:", nir.shape)

## 4. LST pipeline (same math as `src/lst.py` in the project repo)
Landsat Collection 2 Level-2 delivers Band 10 already scaled to surface reflectance/temperature-ready units, so scale factors differ slightly from raw L1 DN — this cell uses the correct C2 L2 scale factors.

In [ ]:
# Landsat Collection 2 Level-2 scale factors (from USGS product guide)
def dn_to_radiance_c2(dn, mult=0.00341802, add=149.0):
    # C2 L2 thermal band is delivered as Kelvin surface temp directly via these factors
    return dn.astype(np.float64) * mult + add

def dn_to_reflectance_c2(dn, mult=0.0000275, add=-0.2):
    return np.clip(dn.astype(np.float64) * mult + add, 0, 1)

# Band 10 in C2 L2 (ST_B10 product) is already surface temperature in Kelvin, scaled
lst_k = dn_to_radiance_c2(band10_dn)
lst_c = lst_k - 273.15

red_refl = dn_to_reflectance_c2(red)
nir_refl = dn_to_reflectance_c2(nir)

with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (nir_refl - red_refl) / (nir_refl + red_refl)

valid = lst_c[np.isfinite(lst_c)]
print(f"LST range: {valid.min():.1f}C to {valid.max():.1f}C, mean {valid.mean():.1f}C")
print(f"NDVI range: {np.nanmin(ndvi):.2f} to {np.nanmax(ndvi):.2f}")

## 5. Quick visual check

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(lst_c, cmap="inferno")
axes[0].set_title("Land Surface Temperature (C)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.6)
axes[1].set_title("NDVI")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

## 6. Fetch Maricopa County census tracts + save LST raster

In [ ]:
import requests, zipfile, io

url = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_04_tract.zip"
resp = requests.get(url)
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("tracts_az")

import geopandas as gpd
tracts = gpd.read_file("tracts_az/tl_2023_04_tract.shp")
maricopa = tracts[tracts["COUNTYFP"] == "013"].to_crs(profile10["crs"])
print(f"Maricopa County tracts: {len(maricopa)}")

# Save LST as GeoTIFF for zonal stats
profile_out = profile10.copy()
profile_out.update(dtype="float32", count=1, nodata=np.nan)
with rasterio.open("lst_phoenix.tif", "w", **profile_out) as dst:
    dst.write(lst_c.astype("float32"), 1)

print("Saved lst_phoenix.tif")

## 7. Zonal stats: mean LST per tract (clipped to scene extent)

In [ ]:
from rasterstats import zonal_stats

scene_bounds = rasterio.open("lst_phoenix.tif").bounds
from shapely.geometry import box
scene_box = box(*scene_bounds)
tracts_in_scene = maricopa[maricopa.intersects(scene_box)].copy()

stats = zonal_stats(tracts_in_scene, "lst_phoenix.tif", stats=["mean", "std", "min", "max"], nodata=np.nan)
import pandas as pd
stats_df = pd.DataFrame(stats).add_prefix("lst_")
tracts_in_scene = pd.concat([tracts_in_scene.reset_index(drop=True), stats_df], axis=1)

print(f"{len(tracts_in_scene)} tracts covered by this scene")
tracts_in_scene[["GEOID", "lst_mean", "lst_min", "lst_max"]].sort_values("lst_mean", ascending=False).head(10)

## 8. Download results to your phone

In [ ]:
tracts_in_scene.drop(columns="geometry").to_csv("phoenix_lst_by_tract.csv", index=False)
tracts_in_scene.to_file("phoenix_lst_tracts.geojson", driver="GeoJSON")

from google.colab import files
files.download("phoenix_lst_by_tract.csv")
files.download("phoenix_lst_tracts.geojson")
files.download("lst_phoenix.tif")

## Next steps
Send the downloaded `phoenix_lst_by_tract.csv` / `.geojson` back — next we add canopy + demographic data, build the vulnerability index, and run Moran's I hotspot clustering.